# 5 · Predict molecule locations
Before starting, read [README.md](README.md). Run cells in order. Example files are from DECODE’s experiment, not the Nanoimager. Do not interpret a successful run as validated microscopy results. Nothing trains or writes data unless you explicitly enable it.
First fit a small subset of the bundled example using its model and matching camera metadata. Then use your own raw movie as an exploratory test only: the example model is not validated for your microscope. For each channel, use a suitable calibration/model and camera metadata. The example is single-channel; your data is two halves of one camera frame.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile

# Change this on the lab PC. It must contain experimental_data_workflow/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'experimental_data_workflow').exists() and (ROOT.parent / 'experimental_data_workflow').exists():
    ROOT = ROOT.parent
EXAMPLE = ROOT / 'experimental_data_workflow'
WORK = ROOT / 'decode_work'
print('Project:', ROOT, '| Example exists:', EXAMPLE.exists())


In [ ]:
import torch, yaml, decode
import decode.utils.param_io
import decode.neuralfitter
import decode.neuralfitter.models
import decode.neuralfitter.utils.processing
import decode.neuralfitter.frame_processing
import decode.neuralfitter.scale_transform
import decode.neuralfitter.coord_transform
import decode.neuralfitter.post_processing
import decode.simulation.camera
import decode.utils.model_io

USE_EXAMPLE = True
RAW_TIFF = None  # set a raw TIFF path when USE_EXAMPLE = False
CHANNEL = 'left' # left or right; raw split relative to image midpoint
FIRST_FRAME = 0
N_FRAMES = 20  # keep small until you understand memory requirements
MIRROR = True  # official example notebook mirrors x; verify orientation for YOUR calibration
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model_path = EXAMPLE / 'model.pt'
param_path = EXAMPLE / 'param_run.yaml'


### Read a small frame range
The raw right-channel coordinate origin becomes the middle of the original frame; save that offset when combining channels. DECODE’s centre crop may remove border pixels, which also changes the origin. The example uses camera metadata from `frames_meta.yaml`; your camera values must be measured before own-data predictions.


In [ ]:
if FIRST_FRAME < 0 or N_FRAMES < 1: raise ValueError('Choose a valid positive frame range.')
path = EXAMPLE / 'frames.tif' if USE_EXAMPLE else Path(RAW_TIFF)
with tifffile.TiffFile(path) as tif:
    if FIRST_FRAME + N_FRAMES > len(tif.pages): raise ValueError('Requested frame range exceeds TIFF pages.')
    arr = np.stack([tif.pages[i].asarray() for i in range(FIRST_FRAME, FIRST_FRAME + N_FRAMES)])
if arr.ndim != 3: raise ValueError('Expected grayscale frames.')
x_offset = 0
if not USE_EXAMPLE:
    if CHANNEL not in ('left', 'right') or arr.shape[2] % 2: raise ValueError('Choose left/right and a TIFF with two equal halves.')
    width = arr.shape[2] // 2
    if CHANNEL == 'left': arr = arr[:, :, :width]
    else: arr, x_offset = arr[:, :, width:], width
print('Frames:', arr.shape, 'right-half raw x offset:', x_offset)
frames = torch.from_numpy(arr.copy()).float()
plt.imshow(arr[0], cmap='gray'); plt.title('Selected channel, first frame'); plt.colorbar(label='ADU'); plt.show()


### Camera conversion and model
For the example, load its separate recorded-frame metadata (EM gain differs from the training YAML). For your own camera, replace the placeholders with measured settings, including sCMOS `em_gain=None`; do not run the own-data cell until you have them. Values must match DECODE’s conventions. The model’s pixel size and scaling remain those in its training parameters.


In [ ]:
param = decode.utils.param_io.load_params(str(param_path))
model = decode.neuralfitter.models.SigmaMUNet.parse(param)
model = decode.utils.model_io.LoadSaveModel(model, input_file=model_path, output_file=None).load_init(device=DEVICE)
model = model.to(DEVICE)
if USE_EXAMPLE:
    with (EXAMPLE / 'frames_meta.yaml').open() as f: camera_meta = yaml.safe_load(f)['Camera']
else:
    camera_meta = {'baseline': None, 'e_per_adu': None, 'em_gain': None, 'spur_noise': None}
    if any(camera_meta[k] is None for k in ('baseline', 'e_per_adu', 'spur_noise')):
        raise ValueError('Replace camera_meta placeholders with measured Nanoimager settings before proceeding.')
param = decode.utils.param_io.RecursiveNamespace(**decode.utils.param_io.autofill_dict(camera_meta, param.to_dict(), mode_missing='include'))
camera = decode.simulation.camera.Photon2Camera.parse(param)
camera.device = 'cpu'


In [ ]:
processing = decode.neuralfitter.utils.processing
steps = [processing.wrap_callable(camera.backward), decode.neuralfitter.frame_processing.AutoCenterCrop(8)]
if MIRROR: steps.append(decode.neuralfitter.frame_processing.Mirror2D(dims=-1))
steps.append(decode.neuralfitter.scale_transform.AmplitudeRescale.parse(param))
frame_proc = processing.TransformSequence(steps)
shape = decode.neuralfitter.frame_processing.get_frame_extent(frames.unsqueeze(1).size(), frame_proc.forward)
extent = ((-0.5, shape[-2]-0.5), (-0.5, shape[-1]-0.5))
print('Original H,W:', arr.shape[-2:], 'processed H,W:', shape[-2:])
print('Centre crop offsets (top,left):', ((arr.shape[-2]-shape[-2])//2, (arr.shape[-1]-shape[-1])//2), '| mirrored:', MIRROR)
post = processing.TransformSequence([
    decode.neuralfitter.scale_transform.InverseParamListRescale.parse(param),
    decode.neuralfitter.coord_transform.Offset2Coordinate(xextent=extent[0], yextent=extent[1], img_shape=shape[-2:]),
    decode.neuralfitter.post_processing.SpatialIntegration(raw_th=0.1, xy_unit='px', px_size=param.Camera.px_size)])
infer = decode.neuralfitter.Infer(model=model, ch_in=param.HyperParameter.channels_in, frame_proc=frame_proc, post_proc=post, device=DEVICE, num_workers=0)
emitter = infer.forward(frames)
print(emitter)


Output coordinates are in the **processed** frame coordinate system, not automatically registered between channels or mapped back to the original movie. Record the crop, mirroring, right-half offset and `FIRST_FRAME`; validate the exact coordinate convention against known spots before transforming. Registration and drift corrections should normally be applied consistently to localisations and assessed independently. No drift correction or automatic mapping is performed here. HDF5 export below is opt-in.


In [ ]:
SAVE_OUTPUT = False
if SAVE_OUTPUT:
    WORK.mkdir(parents=True, exist_ok=True)
    output = WORK / 'example_emitters.h5'  # change for each dataset/channel
    if output.exists(): raise FileExistsError(output)
    emitter.save(str(output))
    print('Saved:', output, '; frame indices start at selected frame 0, original start:', FIRST_FRAME)
else:
    print('Set SAVE_OUTPUT = True after reviewing the coordinates and filename.')
